# Stage 2 v2 local FP32 — ViT-B/16 focal loss
Notebook chạy trực tiếp trên Windows/RTX 4050 bằng FP32. Chỉ chạy một notebook local tại một thời điểm; external test luôn bị khóa.

In [1]:
from pathlib import Path
import os, sys

# Nếu Jupyter không được mở từ repository, đặt ví dụ:
# LOCAL_REPO_ROOT = Path(r"D:\Junior\ĐATH\main")
LOCAL_REPO_ROOT = None

def discover_repo_root(explicit_root=None):
    candidates = []
    if explicit_root is not None:
        candidates.append(Path(explicit_root).expanduser().resolve())
    environment_root = os.environ.get("DATH_REPO_ROOT")
    if environment_root:
        candidates.append(Path(environment_root).expanduser().resolve())
    current = Path.cwd().resolve()
    candidates.extend([current, *current.parents])
    for candidate in candidates:
        if (candidate / "configs" / "default.yaml").is_file() and (candidate / "src").is_dir():
            return candidate
    raise FileNotFoundError("Không tìm thấy repository. Hãy đặt LOCAL_REPO_ROOT bằng Path raw-string.")

REPO_DIR = discover_repo_root(LOCAL_REPO_ROOT)
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

LOCAL_STAGE_ID = "stage2_focalloss_v2_local_fp32_vit_b16"
MODEL_NAME = "vit_b_16"
MODEL_SLUG = "vit_b_16"
BASE_CONFIG = "configs/vit_aug_auprc.yaml"
WEIGHTING = "sqrtcap3"
POSITIVE_WEIGHT = {"mode": "sqrt_ratio_capped", "max_value": 3.0}
RUN_FINAL_TEST = False
MAX_NEW_RUNS_PER_SESSION = None
TRAIN_BATCH_SIZE = 8
VALIDATION_BATCH_SIZE = 16
NUM_WORKERS = 0
PREFETCH_FACTOR = 2

PROCESSED_ROOT = REPO_DIR / "data" / "processed" / "consensus_v1_224"
PROTOCOL_ROOT = REPO_DIR / "data" / "protocols" / "vindr_s8515_exttest_v1"
CHECKPOINT_ROOT = REPO_DIR / "checkpoint" / LOCAL_STAGE_ID / MODEL_SLUG
LOCAL_STAGE_ROOT = REPO_DIR / "outputs" / LOCAL_STAGE_ID
RUN_LOG_ROOT = LOCAL_STAGE_ROOT / MODEL_SLUG
RUNS_ROOT = RUN_LOG_ROOT / "runs"
ANALYSIS_ROOT = RUN_LOG_ROOT / "analysis"
HARDWARE_ROOT = RUN_LOG_ROOT / "hardware"
RUNTIME_CONFIG_ROOT = LOCAL_STAGE_ROOT / "runtime_configs" / MODEL_SLUG
EXPORT_PATH = REPO_DIR / "outputs" / "exports" / "04_3_local_vit_b16_stage2_focalloss_v2_fp32_results.zip"

assert RUN_FINAL_TEST is False
print("Repository:", REPO_DIR)
print("Processed data:", PROCESSED_ROOT)
print("Checkpoint root:", CHECKPOINT_ROOT)
print("Run-log root:", RUN_LOG_ROOT)

Repository: D:\Junior\ĐATH\main
Processed data: D:\Junior\ĐATH\main\data\processed\consensus_v1_224
Checkpoint root: D:\Junior\ĐATH\main\checkpoint\stage2_focalloss_v2_local_fp32_vit_b16\vit_b_16
Run-log root: D:\Junior\ĐATH\main\outputs\stage2_focalloss_v2_local_fp32_vit_b16\vit_b_16


In [2]:
import json, platform, subprocess
import torch, torchvision
from torchvision.models import ViT_B_16_Weights

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
if not torch.cuda.is_available():
    raise RuntimeError("Local Stage 2 yêu cầu CUDA GPU")
gpu_name = torch.cuda.get_device_name(0)
gpu_memory_mib = int(torch.cuda.get_device_properties(0).total_memory / 1024 ** 2)
if "RTX 4050" not in gpu_name.upper():
    raise RuntimeError(f"Expected RTX 4050 Laptop GPU, got {gpu_name}")
if gpu_memory_mib < 5500:
    raise RuntimeError(f"Expected at least 5.5 GB VRAM, got {gpu_memory_mib} MiB")
if torch.__version__ != "2.7.0+cu126" or torchvision.__version__ != "0.22.0+cu126":
    raise RuntimeError(f"Use .venv-1 / requirements.txt; got torch={torch.__version__}, torchvision={torchvision.__version__}")
fp32_probe = torch.ones(8, 8, device="cuda", dtype=torch.float32) @ torch.ones(8, 8, device="cuda", dtype=torch.float32)
if not torch.isfinite(fp32_probe).all():
    raise RuntimeError("Local CUDA FP32 probe failed")
del fp32_probe
torch.cuda.empty_cache()
pretrained_state = ViT_B_16_Weights.DEFAULT.get_state_dict(progress=True, check_hash=True)
if not pretrained_state or not all(torch.isfinite(value).all().item() for value in pretrained_state.values() if torch.is_tensor(value)):
    raise RuntimeError("ViT-B/16 pretrained state is missing or nonfinite")
del pretrained_state
smi = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader,nounits"], check=True, capture_output=True, text=True, encoding="utf-8", errors="replace").stdout.strip()
hardware_profile = {
    "runtime_mode": "local_windows", "platform": platform.platform(),
    "cpu_count": os.cpu_count(), "gpu_name": gpu_name,
    "gpu_memory_mib": gpu_memory_mib, "nvidia_smi": smi,
    "torch": torch.__version__, "torchvision": torchvision.__version__,
    "cuda": torch.version.cuda, "cudnn": torch.backends.cudnn.version(),
    "train_batch_size": TRAIN_BATCH_SIZE, "validation_batch_size": VALIDATION_BATCH_SIZE,
    "num_workers": NUM_WORKERS, "prefetch_factor": PREFETCH_FACTOR if NUM_WORKERS > 0 else None,
    "amp": False, "precision": "float32",
    "cudnn_benchmark": False, "cudnn_deterministic": True,
}
print(json.dumps(hardware_profile, indent=2))
print("ViT-B/16 pretrained + CUDA FP32 preflight: OK")

{
  "runtime_mode": "local_windows",
  "platform": "Windows-11-10.0.26300-SP0",
  "cpu_count": 16,
  "gpu_name": "NVIDIA GeForce RTX 4050 Laptop GPU",
  "gpu_memory_mib": 6140,
  "nvidia_smi": "NVIDIA GeForce RTX 4050 Laptop GPU, 6141, 610.62",
  "torch": "2.7.0+cu126",
  "torchvision": "0.22.0+cu126",
  "cuda": "12.6",
  "cudnn": 90701,
  "train_batch_size": 8,
  "validation_batch_size": 16,
  "num_workers": 0,
  "prefetch_factor": null,
  "amp": false,
  "precision": "float32",
  "cudnn_benchmark": false,
  "cudnn_deterministic": true
}
ViT-B/16 pretrained + CUDA FP32 preflight: OK


In [3]:
import copy, yaml
from src.analysis.stage2 import analyze_stage2
from src.experiments.stage2_colab import _validate_train_data
from src.experiments.stage2_colab_v2 import (
    _run_completion, _run_subprocess_streaming, _start_telemetry, _stop_telemetry,
    build_stage2_experiments, create_or_validate_contract,
    validate_export_members, write_json_with_retry,
)
from src.utils.checksums import file_sha256
from src.utils.config import load_config

for path in (CHECKPOINT_ROOT, RUNS_ROOT, ANALYSIS_ROOT, HARDWARE_ROOT, RUNTIME_CONFIG_ROOT, EXPORT_PATH.parent):
    path.mkdir(parents=True, exist_ok=True)
data_summary = _validate_train_data(PROCESSED_ROOT, PROTOCOL_ROOT)
experiments = build_stage2_experiments(MODEL_NAME, MODEL_SLUG, BASE_CONFIG, WEIGHTING, POSITIVE_WEIGHT)
for experiment in experiments:
    experiment["run_name"] = experiment["run_name"].removesuffix("_amp") + "_fp32"
assert len(experiments) == 4 and len({item["run_name"] for item in experiments}) == 4

def build_local_runtime_config(experiment):
    config = load_config(REPO_DIR / experiment["config"])
    config["data"].update({
        "processed_root": str(PROCESSED_ROOT), "protocol_dir": str(PROTOCOL_ROOT),
        "train_manifest_path": str(PROTOCOL_ROOT / "train_manifest.csv"),
        "val_manifest_path": str(PROTOCOL_ROOT / "val_manifest.csv"),
    })
    config["training"].update({
        "batch_size": TRAIN_BATCH_SIZE, "validation_batch_size": VALIDATION_BATCH_SIZE,
        "num_workers": NUM_WORKERS, "persistent_workers": NUM_WORKERS > 0,
        "prefetch_factor": PREFETCH_FACTOR, "use_amp": False,
        "show_progress": False, "monitor": "macro_average_precision",
        "loss": copy.deepcopy(experiment["loss"]),
    })
    runtime_dir = RUNTIME_CONFIG_ROOT / experiment["run_name"]
    runtime_dir.mkdir(parents=True, exist_ok=True)
    runtime_path = runtime_dir / "default.yaml"
    runtime_path.write_text(yaml.safe_dump(config, sort_keys=False, allow_unicode=True), encoding="utf-8")
    return runtime_path, load_config(runtime_path)

runtime_configs = {item["run_name"]: build_local_runtime_config(item) for item in experiments}
for experiment in experiments:
    (CHECKPOINT_ROOT / experiment["run_name"]).mkdir(parents=True, exist_ok=True)
    result_dir = RUNS_ROOT / experiment["run_name"]
    result_dir.mkdir(parents=True, exist_ok=True)
    status_path = result_dir / "run_status.json"
    if not status_path.exists():
        write_json_with_retry(status_path, {"state": "pending", "run_name": experiment["run_name"]})
source_files = [REPO_DIR / "src/losses/classification_loss.py", REPO_DIR / "src/training/engine.py", REPO_DIR / "scripts/train.py", REPO_DIR / "src/experiments/stage2_colab_v2.py"]
source_sha256 = {str(path.relative_to(REPO_DIR)): file_sha256(path) for path in source_files}
contract = {
    "stage_id": LOCAL_STAGE_ID, "runtime_mode": "local_windows",
    "hardware_profile": hardware_profile, "data_summary": data_summary,
    "source_sha256": source_sha256, "experiments": experiments,
    "selection_metric": "macro_average_precision", "external_test_used": False,
}
create_or_validate_contract(LOCAL_STAGE_ROOT / "stage2_contract.json", contract, allow_create=True)
write_json_with_retry(HARDWARE_ROOT / "hardware_profile.json", hardware_profile)
write_json_with_retry(RUN_LOG_ROOT / "experiment_registry.json", experiments)
print([item["run_name"] for item in experiments])
print("LOCAL VIT-B/16 STAGE 2 FP32 READY")

d:\Junior\ĐATH\main\.venv-1\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


['vit_b_16_stage2v2_bce_sqrtcap3_aug_auprc_s8515_exttest_v1_seed42_fp32', 'vit_b_16_stage2v2_focal_g05_sqrtcap3_aug_auprc_s8515_exttest_v1_seed42_fp32', 'vit_b_16_stage2v2_focal_g10_sqrtcap3_aug_auprc_s8515_exttest_v1_seed42_fp32', 'vit_b_16_stage2v2_focal_g15_sqrtcap3_aug_auprc_s8515_exttest_v1_seed42_fp32']
LOCAL VIT-B/16 STAGE 2 FP32 READY


In [4]:
import subprocess, traceback
from datetime import datetime, timezone

def utc_now():
    return datetime.now(timezone.utc).isoformat()

new_runs = 0
for index, experiment in enumerate(experiments, start=1):
    config_path, config = runtime_configs[experiment["run_name"]]
    checkpoint_dir = CHECKPOINT_ROOT / experiment["run_name"]
    result_dir = RUNS_ROOT / experiment["run_name"]
    complete, resume, stop_reason = _run_completion(experiment, config, CHECKPOINT_ROOT, amp_enabled=False)
    status_path = result_dir / "run_status.json"
    if complete:
        write_json_with_retry(status_path, {"state": "complete", "updated_at_utc": utc_now(), "stop_reason": stop_reason, "run_name": experiment["run_name"]})
        print(f"[{index}/4] skip complete: {experiment['run_name']}")
    else:
        if MAX_NEW_RUNS_PER_SESSION is not None and new_runs >= MAX_NEW_RUNS_PER_SESSION:
            break
        write_json_with_retry(status_path, {"state": "resuming" if resume else "running", "updated_at_utc": utc_now(), "run_name": experiment["run_name"]})
        command = [sys.executable, str(REPO_DIR / "scripts" / "train.py"), "--config", str(config_path), "--checkpoint-dir", str(checkpoint_dir), "--result-dir", str(result_dir)]
        if resume:
            command.extend(["--resume", str(checkpoint_dir / "last.pt")])
        environment = os.environ.copy()
        environment["PYTHONPATH"] = str(REPO_DIR)
        environment["PYTHONUNBUFFERED"] = "1"
        environment["PYTHONIOENCODING"] = "utf-8"
        environment["PYTHONUTF8"] = "1"
        local_subprocess_root = LOCAL_STAGE_ROOT / "subprocess_logs"
        local_log_path = local_subprocess_root / f"{result_dir.name}.log"
        telemetry_process, telemetry_handle = _start_telemetry(HARDWARE_ROOT / f"gpu_telemetry_{experiment['run_name']}.csv")
        try:
            return_code, output_tail = _run_subprocess_streaming(command, REPO_DIR, environment, result_dir, local_root=local_subprocess_root)
        except BaseException as error:
            log_tail = "" if not local_log_path.is_file() else "\n".join(local_log_path.read_text(encoding="utf-8", errors="replace").splitlines()[-200:])
            failure = {"state": "interrupted" if isinstance(error, KeyboardInterrupt) else "failed", "updated_at_utc": utc_now(), "run_name": experiment["run_name"], "error_type": type(error).__name__, "error": str(error), "traceback": traceback.format_exc()[-12000:], "traceback_tail": log_tail, "log_file": str(local_log_path)}
            write_json_with_retry(result_dir / "training_failure.json", failure)
            write_json_with_retry(status_path, failure)
            raise
        finally:
            try:
                _stop_telemetry(telemetry_process, telemetry_handle)
            except Exception as telemetry_error:
                print(f"Warning: telemetry cleanup failed: {telemetry_error}")
        if return_code != 0:
            failure = {"state": "failed", "updated_at_utc": utc_now(), "run_name": experiment["run_name"], "return_code": return_code, "traceback_tail": output_tail, "log_file": "train_subprocess.log"}
            write_json_with_retry(result_dir / "training_failure.json", failure)
            write_json_with_retry(status_path, failure)
            raise RuntimeError(f"Training failed: {experiment['run_name']}\n{output_tail}")
        complete, _, stop_reason = _run_completion(experiment, config, CHECKPOINT_ROOT, amp_enabled=False)
        if not complete:
            failure = {"state": "failed", "updated_at_utc": utc_now(), "run_name": experiment["run_name"], "error": "Training exited without epoch-limit or early-stopping completion"}
            write_json_with_retry(result_dir / "training_failure.json", failure)
            write_json_with_retry(status_path, failure)
            raise RuntimeError(failure["error"])
        write_json_with_retry(status_path, {"state": "complete", "updated_at_utc": utc_now(), "stop_reason": stop_reason, "run_name": experiment["run_name"]})
        (result_dir / "training_failure.json").unlink(missing_ok=True)
        new_runs += 1
    partial = analyze_stage2(experiments, RUN_LOG_ROOT, CHECKPOINT_ROOT, ANALYSIS_ROOT)
    if not partial["macro"].empty:
        display(partial["macro"])
print("Local ViT-B/16 Stage 2 training pass finished")

{'epoch': 1, 'train_loss': 0.30555232455566816, 'val_loss': 0.26950411836306254, 'val_macro_auroc': 0.7967531654144119, 'val_macro_ap': 0.13946217875003994, 'val_macro_f1_tuned': 0.20471173791077682, 'monitor_name': 'macro_average_precision', 'monitor_score': 0.13946217875003994, 'threshold_min': 0.0380045622587204, 'threshold_mean': 0.16969868301280908, 'threshold_max': 0.5493212342262268, 'train_seconds': 429.54800579999574, 'val_seconds': 22.061611600016477, 'train_images_per_second': 29.682363386262814, 'val_images_per_second': 101.9871095907753, 'peak_gpu_memory_gb': 1.9491534233093262, 'peak_gpu_memory_allocated_gb': 1.9491534233093262, 'peak_gpu_memory_reserved_gb': 2.2890625}
{'epoch': 2, 'train_loss': 0.2711409093235053, 'val_loss': 0.2571724146472083, 'val_macro_auroc': 0.8650083966382525, 'val_macro_ap': 0.21428489756869468, 'val_macro_f1_tuned': 0.28082182198783817, 'monitor_name': 'macro_average_precision', 'monitor_score': 0.21428489756869468, 'threshold_min': 0.051816716

,run_name,model,variant,loss_name,gamma,weighting,status,best_epoch,macro_average_precision,macro_precision,...,checkpoint_sha256,precision_at_control_recall,classes_ap_improved,classes_ap_decreased,classes_ap_drop_gt_002,delta_macro_average_precision_vs_bce,delta_macro_precision_vs_bce,delta_macro_recall_vs_bce,delta_macro_f1_vs_bce,eligible
0,vit_b_16_stage2v2_bce_sqrtcap3_aug_auprc_s8515...,vit_b_16,bce,weighted_bce,None,sqrtcap3,complete,21,0.372035,0.376571,...,a4f5221392a91a8657ea5194cb43b30ceb0157b1b780c8...,0.376571,0.0,0.0,0.0,0.0,0.0,0.0,0.0,True


{'epoch': 1, 'train_loss': 0.21748057572631274, 'val_loss': 0.1883065698477957, 'val_macro_auroc': 0.8240085785045786, 'val_macro_ap': 0.1605692290422896, 'val_macro_f1_tuned': 0.22178864530179668, 'monitor_name': 'macro_average_precision', 'monitor_score': 0.1605692290422896, 'threshold_min': 0.05534566193819046, 'threshold_mean': 0.20570221543312073, 'threshold_max': 0.5047881007194519, 'train_seconds': 478.6600470000121, 'val_seconds': 27.990363199991407, 'train_images_per_second': 26.636858622127026, 'val_images_per_second': 80.38480901172053, 'peak_gpu_memory_gb': 1.9491539001464844, 'peak_gpu_memory_allocated_gb': 1.9491539001464844, 'peak_gpu_memory_reserved_gb': 2.2890625}
{'epoch': 2, 'train_loss': 0.1930061800550012, 'val_loss': 0.17813665180736118, 'val_macro_auroc': 0.8666130964277502, 'val_macro_ap': 0.19815876814905123, 'val_macro_f1_tuned': 0.2718615398715257, 'monitor_name': 'macro_average_precision', 'monitor_score': 0.19815876814905123, 'threshold_min': 0.101148806512

,run_name,model,variant,loss_name,gamma,weighting,status,best_epoch,macro_average_precision,macro_precision,...,checkpoint_sha256,precision_at_control_recall,classes_ap_improved,classes_ap_decreased,classes_ap_drop_gt_002,delta_macro_average_precision_vs_bce,delta_macro_precision_vs_bce,delta_macro_recall_vs_bce,delta_macro_f1_vs_bce,eligible
0,vit_b_16_stage2v2_bce_sqrtcap3_aug_auprc_s8515...,vit_b_16,bce,weighted_bce,NaN,sqrtcap3,complete,21,0.372035,0.376571,...,a4f5221392a91a8657ea5194cb43b30ceb0157b1b780c8...,0.376571,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,True
1,vit_b_16_stage2v2_focal_g05_sqrtcap3_aug_auprc...,vit_b_16,focal_g05,sigmoid_focal,0.5,sqrtcap3,complete,23,0.384349,0.411875,...,3596e0960d8d5579b0680df4b2ae355c5470909005f821...,0.321652,8.0,6.0,3.0,0.012314,0.035304,-0.021999,0.017031,False


{'epoch': 1, 'train_loss': 0.16028899480022635, 'val_loss': 0.13438229432702065, 'val_macro_auroc': 0.8217899069166963, 'val_macro_ap': 0.1507998404643954, 'val_macro_f1_tuned': 0.221071414003036, 'monitor_name': 'macro_average_precision', 'monitor_score': 0.1507998404643954, 'threshold_min': 0.11497563868761063, 'threshold_mean': 0.2666603097958224, 'threshold_max': 0.5261472463607788, 'train_seconds': 460.76930779998656, 'val_seconds': 22.696760399994673, 'train_images_per_second': 27.671113904866672, 'val_images_per_second': 99.13309037709752, 'peak_gpu_memory_gb': 1.9491539001464844, 'peak_gpu_memory_allocated_gb': 1.9491539001464844, 'peak_gpu_memory_reserved_gb': 2.2890625}
{'epoch': 2, 'train_loss': 0.14407049592686635, 'val_loss': 0.1415915925502777, 'val_macro_auroc': 0.8293675639758468, 'val_macro_ap': 0.170965593628379, 'val_macro_f1_tuned': 0.240884935595954, 'monitor_name': 'macro_average_precision', 'monitor_score': 0.170965593628379, 'threshold_min': 0.14525999128818512,

,run_name,model,variant,loss_name,gamma,weighting,status,best_epoch,macro_average_precision,macro_precision,...,checkpoint_sha256,precision_at_control_recall,classes_ap_improved,classes_ap_decreased,classes_ap_drop_gt_002,delta_macro_average_precision_vs_bce,delta_macro_precision_vs_bce,delta_macro_recall_vs_bce,delta_macro_f1_vs_bce,eligible
0,vit_b_16_stage2v2_bce_sqrtcap3_aug_auprc_s8515...,vit_b_16,bce,weighted_bce,NaN,sqrtcap3,complete,21,0.372035,0.376571,...,a4f5221392a91a8657ea5194cb43b30ceb0157b1b780c8...,0.376571,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,True
1,vit_b_16_stage2v2_focal_g05_sqrtcap3_aug_auprc...,vit_b_16,focal_g05,sigmoid_focal,0.5,sqrtcap3,complete,23,0.384349,0.411875,...,3596e0960d8d5579b0680df4b2ae355c5470909005f821...,0.321652,8.0,6.0,3.0,0.012314,0.035304,-0.021999,0.017031,False
2,vit_b_16_stage2v2_focal_g10_sqrtcap3_aug_auprc...,vit_b_16,focal_g10,sigmoid_focal,1.0,sqrtcap3,complete,16,0.345083,0.357263,...,3704fa3a30c77fa66bab0eb9475b3f62ff2dde940798de...,0.295605,4.0,10.0,7.0,-0.026952,-0.019308,-0.028449,-0.020786,False


{'epoch': 1, 'train_loss': 0.11414623006827691, 'val_loss': 0.09684546327259805, 'val_macro_auroc': 0.8261360777846187, 'val_macro_ap': 0.16080337823175528, 'val_macro_f1_tuned': 0.22550844200693138, 'monitor_name': 'macro_average_precision', 'monitor_score': 0.16080337823175528, 'threshold_min': 0.1428881734609604, 'threshold_mean': 0.29208507175956455, 'threshold_max': 0.496232271194458, 'train_seconds': 443.8989361000131, 'val_seconds': 22.661945400002878, 'train_images_per_second': 28.722754129618703, 'val_images_per_second': 99.28538615222831, 'peak_gpu_memory_gb': 1.9491539001464844, 'peak_gpu_memory_allocated_gb': 1.9491539001464844, 'peak_gpu_memory_reserved_gb': 2.2890625}
{'epoch': 2, 'train_loss': 0.09920330854403038, 'val_loss': 0.09796056301063961, 'val_macro_auroc': 0.8665549709726109, 'val_macro_ap': 0.2030430893265533, 'val_macro_f1_tuned': 0.2703349584161134, 'monitor_name': 'macro_average_precision', 'monitor_score': 0.2030430893265533, 'threshold_min': 0.214543238282

,run_name,model,variant,loss_name,gamma,weighting,status,best_epoch,macro_average_precision,macro_precision,...,checkpoint_sha256,precision_at_control_recall,classes_ap_improved,classes_ap_decreased,classes_ap_drop_gt_002,delta_macro_average_precision_vs_bce,delta_macro_precision_vs_bce,delta_macro_recall_vs_bce,delta_macro_f1_vs_bce,eligible
0,vit_b_16_stage2v2_bce_sqrtcap3_aug_auprc_s8515...,vit_b_16,bce,weighted_bce,NaN,sqrtcap3,complete,21,0.372035,0.376571,...,a4f5221392a91a8657ea5194cb43b30ceb0157b1b780c8...,0.376571,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,True
1,vit_b_16_stage2v2_focal_g05_sqrtcap3_aug_auprc...,vit_b_16,focal_g05,sigmoid_focal,0.5,sqrtcap3,complete,23,0.384349,0.411875,...,3596e0960d8d5579b0680df4b2ae355c5470909005f821...,0.321652,8.0,6.0,3.0,0.012314,0.035304,-0.021999,0.017031,False
2,vit_b_16_stage2v2_focal_g10_sqrtcap3_aug_auprc...,vit_b_16,focal_g10,sigmoid_focal,1.0,sqrtcap3,complete,16,0.345083,0.357263,...,3704fa3a30c77fa66bab0eb9475b3f62ff2dde940798de...,0.295605,4.0,10.0,7.0,-0.026952,-0.019308,-0.028449,-0.020786,False
3,vit_b_16_stage2v2_focal_g15_sqrtcap3_aug_auprc...,vit_b_16,focal_g15,sigmoid_focal,1.5,sqrtcap3,complete,22,0.388739,0.451159,...,986101faffbd822d10f8fe2f9bb58dad0e3ad99fa9f64a...,0.386298,7.0,7.0,2.0,0.016704,0.074588,0.000070,0.019546,True


Local ViT-B/16 Stage 2 training pass finished


In [5]:
import shutil, zipfile
analysis = analyze_stage2(experiments, RUN_LOG_ROOT, CHECKPOINT_ROOT, ANALYSIS_ROOT)
display(analysis["macro"])
display(analysis["ranking"])
manifest = {"created_at_utc": utc_now(), "stage_id": LOCAL_STAGE_ID, "runtime_mode": "local_windows", "precision": "float32", "model": MODEL_NAME, "checkpoint_root": str(CHECKPOINT_ROOT), "run_log_root": str(RUN_LOG_ROOT), "selection_metric": "macro_average_precision", "external_test_used": False}
write_json_with_retry(RUN_LOG_ROOT / "experiment_manifest.json", manifest)
archive_base = EXPORT_PATH.with_suffix("")
if EXPORT_PATH.exists():
    EXPORT_PATH.unlink()
shutil.make_archive(str(archive_base), "zip", root_dir=LOCAL_STAGE_ROOT)
with zipfile.ZipFile(EXPORT_PATH) as archive:
    validate_export_members(archive.namelist())
print("Export ready:", EXPORT_PATH)
print("External test locked.")

,run_name,model,variant,loss_name,gamma,weighting,status,best_epoch,macro_average_precision,macro_precision,...,checkpoint_sha256,precision_at_control_recall,classes_ap_improved,classes_ap_decreased,classes_ap_drop_gt_002,delta_macro_average_precision_vs_bce,delta_macro_precision_vs_bce,delta_macro_recall_vs_bce,delta_macro_f1_vs_bce,eligible
0,vit_b_16_stage2v2_bce_sqrtcap3_aug_auprc_s8515...,vit_b_16,bce,weighted_bce,NaN,sqrtcap3,complete,21,0.372035,0.376571,...,a4f5221392a91a8657ea5194cb43b30ceb0157b1b780c8...,0.376571,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,True
1,vit_b_16_stage2v2_focal_g05_sqrtcap3_aug_auprc...,vit_b_16,focal_g05,sigmoid_focal,0.5,sqrtcap3,complete,23,0.384349,0.411875,...,3596e0960d8d5579b0680df4b2ae355c5470909005f821...,0.321652,8.0,6.0,3.0,0.012314,0.035304,-0.021999,0.017031,False
2,vit_b_16_stage2v2_focal_g10_sqrtcap3_aug_auprc...,vit_b_16,focal_g10,sigmoid_focal,1.0,sqrtcap3,complete,16,0.345083,0.357263,...,3704fa3a30c77fa66bab0eb9475b3f62ff2dde940798de...,0.295605,4.0,10.0,7.0,-0.026952,-0.019308,-0.028449,-0.020786,False
3,vit_b_16_stage2v2_focal_g15_sqrtcap3_aug_auprc...,vit_b_16,focal_g15,sigmoid_focal,1.5,sqrtcap3,complete,22,0.388739,0.451159,...,986101faffbd822d10f8fe2f9bb58dad0e3ad99fa9f64a...,0.386298,7.0,7.0,2.0,0.016704,0.074588,0.000070,0.019546,True


,run_name,model,variant,loss_name,gamma,weighting,status,best_epoch,macro_average_precision,macro_precision,...,classes_ap_improved,classes_ap_decreased,classes_ap_drop_gt_002,delta_macro_average_precision_vs_bce,delta_macro_precision_vs_bce,delta_macro_recall_vs_bce,delta_macro_f1_vs_bce,eligible,variant_order,rank
0,vit_b_16_stage2v2_focal_g15_sqrtcap3_aug_auprc...,vit_b_16,focal_g15,sigmoid_focal,1.5,sqrtcap3,complete,22,0.388739,0.451159,...,7.0,7.0,2.0,0.016704,0.074588,0.000070,0.019546,True,3,1
1,vit_b_16_stage2v2_bce_sqrtcap3_aug_auprc_s8515...,vit_b_16,bce,weighted_bce,NaN,sqrtcap3,complete,21,0.372035,0.376571,...,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,True,0,2
2,vit_b_16_stage2v2_focal_g05_sqrtcap3_aug_auprc...,vit_b_16,focal_g05,sigmoid_focal,0.5,sqrtcap3,complete,23,0.384349,0.411875,...,8.0,6.0,3.0,0.012314,0.035304,-0.021999,0.017031,False,1,3
3,vit_b_16_stage2v2_focal_g10_sqrtcap3_aug_auprc...,vit_b_16,focal_g10,sigmoid_focal,1.0,sqrtcap3,complete,16,0.345083,0.357263,...,4.0,10.0,7.0,-0.026952,-0.019308,-0.028449,-0.020786,False,2,4


Export ready: D:\Junior\ĐATH\main\outputs\exports\04_3_local_vit_b16_stage2_focalloss_v2_fp32_results.zip
External test locked.
